---
title: "Core Module: Internal functions and testing"
---

## core

> This is a core library for the ERA5 dataset pipeline. It defines
a few helpful functions such as an API tester to test your API key and connection.

<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Utilities

Some utilities are provided to help you with the ERA5 dataset.

In [0]:
#| echo: false
#| output: asis
show_doc(describe)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L24){target="_blank" style="float:right; font-size:smaller"}

### describe

>      describe (cfg:omegaconf.dictconfig.DictConfig=None)

*Describe the configuration file used by Hydra for the pipeline*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| cfg | DictConfig | None | Configuration file |
| **Returns** | **None** |  |  |

In [0]:
#| echo: false
#| output: asis
show_doc(kelvin_to_celsius)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L79){target="_blank" style="float:right; font-size:smaller"}

### kelvin_to_celsius

>      kelvin_to_celsius (kelvin)

*Convert temperature from Kelvin to Celsius.

Args:
    kelvin (float): Temperature in Kelvin.

Returns:
    float: Temperature in Celsius.*

### A Class for Authenticating Google Drive

We're going to use a class to authenticate and interact with google drive. The goal is to have a simple interface to fetch the healthshed files dynamically from google drive in the pipeline.

In [0]:
#| echo: false
#| output: asis
show_doc(GoogleDriver)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L92){target="_blank" style="float:right; font-size:smaller"}

### GoogleDriver

>      GoogleDriver (json_key_path=None)

*A class to handle Google Drive authentication and file management.
This class uses the PyDrive2 library to authenticate with Google Drive using a service account.

It provides three methods: authenticating the account, getting the drive object, and downloading the healthshed files for madagascar.*

Here's how we use it. The credentials for the data-pipeline service account are
available in the sandbox folder, and the path to said folder is set in the config:

In [ ]:
from hydra import initialize, compose
from omegaconf import OmegaConf

In [ ]:
# unfortunately, we have to use the initialize function to load the config file
# this is because the @hydra decorator does not work with Notebooks very well
# this is a known issue with Hydra: https://gist.github.com/bdsaglam/586704a98336a0cf0a65a6e7c247d248
# 
# just use the relative path from the notebook to the config dir
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

In [ ]:
auth = GoogleDriver(json_key_path=here() / cfg.GOOGLE_DRIVE_AUTH_JSON.path)
drive = auth.get_drive()

Here's how we might check that the healthsheds are accessible in the drive:

In [ ]:
# we're using the madagascar healthshed folder as an example
folder_id = cfg.geographies.madagascar.healthsheds
folder_name = "healthsheds2022.zip"
file_list = drive.ListFile({'q': f" title='{folder_name}' and trashed = false "}).GetList()

for file in file_list:
    print(f"{file['title']} - {file['mimeType']}")

healthsheds2022.zip - application/zip


That being said, we can read in  the healthsheds into geopandas by downloading them to a temp directory. The healthsheds must be a zipped shapefiles package with the files at the root of the zip directory.

In [ ]:
with tempfile.TemporaryDirectory() as temp_dir:
    # Create a temporary directory to store the downloaded file
    zip_path = os.path.join(temp_dir, folder_name)

    # Download file from Google Drive
    file_obj = drive.CreateFile({'id': file_list[0]['id']})
    file_obj.GetContentFile(zip_path)

    # Read shapefile directly from ZIP
    gdf = gpd.read_file(f"zip://{zip_path}")

That works! So now we can patch the class to include this workflow:

In [0]:
#| echo: false
#| output: asis
show_doc(GoogleDriver.read_healthsheds)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L127){target="_blank" style="float:right; font-size:smaller"}

### GoogleDriver.read_healthsheds

>      GoogleDriver.read_healthsheds (healthshed_zip_name)

And to check that it works:

In [ ]:
driver = GoogleDriver(json_key_path=here() / cfg.GOOGLE_DRIVE_AUTH_JSON.path)
drive = driver.get_drive()
healthsheds = driver.read_healthsheds("healthsheds2022.zip")

healthsheds.describe()

,fs_pop,n_uid,n_instat,n_comp,n_shape
count,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000
mean,10493.058930,7.480116,6.318149,1.010484,1.036515
std,12127.817529,7.263235,4.939271,0.112019,0.393120
min,0.000000,1.000000,1.000000,1.000000,1.000000
25%,4344.750000,4.000000,3.000000,1.000000,1.000000
50%,7417.000000,6.000000,5.000000,1.000000,1.000000
75%,12531.250000,9.000000,8.000000,1.000000,1.000000
max,194782.000000,104.000000,62.000000,3.000000,15.000000


## CDS File Handler Type

We're going to make a file handler type to help deal with CDS files. This is to fix [NSAPH-Data-Processing/era5_sandbox#13](https://github.com/NSAPH-Data-Processing/era5_sandbox/issues/13). 

Usually, when you download data, it comes out as a simple .nc file that can be opened with xarray. However, the CDS API has a few different file types that are not .nc files. For example, the ERA5 data is stored in a .grib file format. This is a common format for meteorological data, and it is used by the ECMWF. When a query has multiple variables, sometimes they are downloaded as a .zip file to separat the grib from the netcdf.

So, below, we define a class that can handle the file no matter what the type is. It will check the file type and then use the appropriate method to open it. The class will also have a method to check if the file is a .zip file, and if so, it will unzip it and return the path to the unzipped file.

In [0]:
#| echo: false
#| output: asis
show_doc(ClimateDataFileHandler)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L149){target="_blank" style="float:right; font-size:smaller"}

### ClimateDataFileHandler

>      ClimateDataFileHandler (input_path:str)

*A class to handle file operations for the Climate Data Store (CDS).
This class provides unpack files downloaded from the CDS API. It must be able to
handle the unpacking of files downloaded from the CDS API. This means that
if the file is a basic netcdf, it should be passed to the netcdf handler. If
the file is a zip, it should be handled by the zip handler in temp and the
data returned as required.*

In [ ]:
import xarray as xr
from fastcore.test import test_fail

In [ ]:
eg_file = here() / "data/input/madagascar_2023_10.nc"

# this fails because the nc file downloaded has grib and netcdf in it, so
# xr cannot handle it.
def wont_work(multilayer_file):

    ds = xr.open_dataset(multilayer_file)

test_fail(
    wont_work,
    args=(eg_file)
)

# equivalent to saying try: wont_work(eg_file) Except: some error handling

The above fails because the download contains temperature and precipitation data, which get encoded silently as different formats. Even though it is one file, it contains both grib and netcdf data and is encoded as a .zip file. So we use the class to read it instead:

In [ ]:
handler = ClimateDataFileHandler(eg_file)
handler.prepare()
ds1 = xr.open_dataset(handler.get_dataset("instant"))
ds2 = xr.open_dataset(handler.get_dataset("accum"))

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[1], line 2
      1 handler = ClimateDataFileHandler(eg_file)
----> 2 handler.prepare()
      3 ds1 = xr.open_dataset(handler.get_dataset("instant"))
      4 ds2 = xr.open_dataset(handler.get_dataset("accum"))

Cell In[1], line 46, in ClimateDataFileHandler.prepare(self)
     43     return
     45 if not self.original_path.exists():
---> 46     raise FileNotFoundError(f"{self.original_path} does not exist")
     48 # Detect ZIP by magic number
     49 # chatgpt implementation here; this is a common way to check for zip files
     50 with open(self.original_path, "rb") as f:

FileNotFoundError: /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/data/input/madagascar_2023_10.nc does not exist


FileNotFoundError: /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/data/input/madagascar_2023_10.nc does not exist

In [ ]:
ds1

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 1
----> 1 ds1

NameError: name 'ds1' is not defined


NameError: name 'ds1' is not defined

In [ ]:
ds2

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 1
----> 1 ds2

NameError: name 'ds2' is not defined


NameError: name 'ds2' is not defined

In [ ]:
handler.cleanup()

Great! Let's add a context handler and this can be added to the pipeline.

In [0]:
#| echo: false
#| output: asis
show_doc(ClimateDataFileHandler.__exit__)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L246){target="_blank" style="float:right; font-size:smaller"}

### ClimateDataFileHandler.__exit__

>      ClimateDataFileHandler.__exit__ (exc_type, exc_val, exc_tb)

In [0]:
#| echo: false
#| output: asis
show_doc(ClimateDataFileHandler.__enter__)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L241){target="_blank" style="float:right; font-size:smaller"}

### ClimateDataFileHandler.__enter__

>      ClimateDataFileHandler.__enter__ ()

In [ ]:
with ClimateDataFileHandler(eg_file) as handler:
    ds1 = xr.open_dataset(handler.get_dataset("instant"))
    ds2 = xr.open_dataset(handler.get_dataset("accum"))

    print(ds1)
    print(ds2)

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[1], line 1
----> 1 with ClimateDataFileHandler(eg_file) as handler:
      2     ds1 = xr.open_dataset(handler.get_dataset("instant"))
      3     ds2 = xr.open_dataset(handler.get_dataset("accum"))

Cell In[1], line 5, in __enter__(self)
      3 @patch
      4 def __enter__(self:ClimateDataFileHandler):
----> 5     self.prepare()
      6     return self

Cell In[1], line 46, in ClimateDataFileHandler.prepare(self)
     43     return
     45 if not self.original_path.exists():
---> 46     raise FileNotFoundError(f"{self.original_path} does not exist")
     48 # Detect ZIP by magic number
     49 # chatgpt implementation here; this is a common way to check for zip files
     50 with open(self.original_path, "rb") as f:

FileNotFoundError: /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/data/input/m

FileNotFoundError: /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/data/input/madagascar_2023_10.nc does not exist

## Tests and Main

In `nbdev`, our tests are embedded in the notebook. Whenever you export the notebook, all the cells that are specified to run are run, and hence, the tests are executed. The tests are also exported. This is a great way to ensure that your documentation is always up-to-date. For this module, we're using the [`testAPI()`](https://TinasheMTapera.github.io/era5_sandbox/core.html#testapi) function as our main test.

In [0]:
#| echo: false
#| output: asis
show_doc(testAPI)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L250){target="_blank" style="float:right; font-size:smaller"}

### testAPI

>      testAPI (cfg:omegaconf.dictconfig.DictConfig=None,
>               dataset:str='reanalysis-era5-pressure-levels')

We can see that this API tester tool works with Hydra configuration:

In [ ]:
from hydra import initialize, compose
from omegaconf import OmegaConf

In [ ]:
# unfortunately, we have to use the initialize function to load the config file
# this is because the @hydra decorator does not work with Notebooks very well
# this is a known issue with Hydra: https://gist.github.com/bdsaglam/586704a98336a0cf0a65a6e7c247d248
# 
# just use the relative path from the notebook to the config dir
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

describe(cfg)

This package fetches ERA5 data. The following is the config file used by Hydra for the pipeline:

development_mode: false
CDS_API_KEY:
  path: $HOME/.cdsapirc
GOOGLE_DRIVE_AUTH_JSON:
  path: sandbox/harvard-csph-driveauth-f5f9a2682ecf.json
  healthsheds_id: healthsheds2022.zip
mdg_shapefile: https://data.humdata.org/dataset/26fa506b-0727-4d9d-a590-d2abee21ee22/resource/ed94d52e-349e-41be-80cb-62dc0435bd34/download/mdg_adm_bngrc_ocha_20181031_shp.zip
dataset: reanalysis-era5-single-levels
query:
  geography:
  - madagascar
  - nepal
  product_type: reanalysis
  variable:
  - 2m_dewpoint_temperature
  - 2m_temperature
  - total_precipitation
  - volumetric_soil_water_layer_1
  year:
  - 2009
  - 2010
  - 2011
  - 2012
  - 2013
  - 2014
  - 2015
  - 2016
  - 2017
  - 2018
  - 2019
  - 2020
  - 2021
  - 2022
  - 2023
  - 2024
  month:
  - 1
  - 2
  - 3
  - 4
  - 5
  - 6
  - 7
  - 8
  - 9
  - 10
  - 11
  - 12
  day:
  - 1
  - 2
  - 3
  - 4
  - 5
  - 6
  - 7
  - 8
  - 9
  - 10
  - 11
  - 12


### Importing the Main Function

Important: using `__main__` in nbdev and Hydra is a little bit tricky. We need to define the main function in the module ONLY ONCE and then when we export the notebook to script, we need to add the `nbdev.imports.IN_NOTEBOOK` variable. This way, the main function will only be executed when we run the notebook and not when we import the module.

```python
from nbdev.imports import IN_NOTEBOOK
```

You'll see this listed throughout the notebooks.

In [0]:
#| echo: false
#| output: asis
show_doc(main)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L321){target="_blank" style="float:right; font-size:smaller"}

### main

>      main (cfg:omegaconf.dictconfig.DictConfig)